In [1]:
from ipyleaflet import Map, TileLayer, LayersControl, basemaps, basemap_to_tiles, WidgetControl, LegendControl, GeoJSON
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display
import IPython
import requests
import geojson
import json

In [2]:
# Base host url InVEST is expected to be serving
host_url = 'http://localhost:8081'
# Load information dynamically created by most recent schisto model run.
nb_json_config_path = f'{host_url}/nb-json-config.json'

# Get the layers from the most recent model run to display
nb_json_config = requests.get(nb_json_config_path).json()

# Get the plot png paths from the most recent model run
plot_root_path = f'{host_url}/plot-previews/'
plot_png_list = [plot_root_path + plot_name for plot_name in nb_json_config['plot_paths']]

# Get map center
map_center = nb_json_config['aoi_center']
# Get aoi geojson
aoi_geojson_path = f"{host_url}/{nb_json_config['aoi_geojson']}"

In [3]:
empty_colorbar = [(0, "#a4a6a5"), (1.0, "#a4a6a5")]

# Build colorbars for legend
colorbar_list = [empty_colorbar]
colorbar_key = ['empty']
for tile_dir_key, layer_dict in nb_json_config['layers'].items():
    local_colorbar = []
    for interval_key, hex_color in layer_dict['color_profile_hex'].items():
        local_colorbar.append((float(interval_key), hex_color))
    colorbar_list.append(local_colorbar)
    colorbar_key.append(tile_dir_key)

# LEGEND CONTROL
# This method is taken from https://github.com/jupyter-widgets/ipyleaflet/issues/706
import matplotlib.pyplot as plt; import matplotlib as mpl
import io
colorbar_dict = {}
#for key, colorbar in zip(['hab', 'pop', 'empty'], [hab_suit_colorbar, pop_suit_colorbar, empty_colorbar]):
for key, colorbar in zip(colorbar_key, colorbar_list):
    fig, ax = plt.subplots(figsize=(6, 1), layout='constrained')
    norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
    my_cmap = mpl.colors.LinearSegmentedColormap.from_list('mymap', colorbar)
    fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=my_cmap), cax=ax, orientation='horizontal',
        label='Risk', ticks=[x[0] for x in colorbar])
    f = io.BytesIO()
    plt.savefig(f, bbox_inches='tight', format='png')
    image = f.getvalue()
    colorbar_dict[key] = image
    plt.close()


In [4]:
# Create a GridBox from the plot widgets
plot_widgets = []
for plot_image in plot_png_list:
    image = IPython.display.Image(plot_image)
    tmp_widget = widgets.Image(
        value=image.data,
        format='png',
    )
    plot_widgets.append(tmp_widget)

plot_grid = widgets.GridBox(plot_widgets, layout=widgets.Layout(grid_template_columns="repeat(3, 33%)", overflow_x='auto'))

In [5]:
# Set up the leaflet map
chosen_basemap = basemap_to_tiles(basemaps.OpenStreetMap.Mapnik)
chosen_basemap.name = '(basemap) OpenStreetMap Mapnik'
# Need to find a way to programmatically get center for diff locations
m = Map(center=map_center, zoom=8, scroll_wheel_zoom=True, layout=Layout(height='600px'))

In [7]:
# Group layers for easier visualization in map control
group_to_layers = {}
# Map layer names to TileLayer representation
checkbox_layers_map = {}
checkbox_legend_map = {}

for tile_dir_key, layer_dict in nb_json_config['layers'].items():
    group_name = layer_dict['group_name']
    if group_name not in group_to_layers:
        group_to_layers[group_name] = []
    group_to_layers[group_name].append(layer_dict['display_name'])
    base_tile_dir = f"{layer_dict['tile_dir']}_tiles"
    tile_url = TileLayer(
        url=f'{host_url}/{base_tile_dir}/{{z}}/{{x}}/{{y}}.png',
        name=f"{layer_dict['display_name']}",
        attribution="mine", #TODO: do we need anything here?
        min_zoom=1,
        max_zoom=18,
        min_native_zoom=1,
        max_native_zoom=14,)
    checkbox_layers_map[layer_dict['display_name']] = tile_url
    checkbox_legend_map[layer_dict['display_name']] = tile_dir_key

In [8]:
%%capture
# Add AOI layer
aoi_geojson = requests.get(aoi_geojson_path).json()
aoi_key = 'area_of_interest'
geojson_layer = GeoJSON(
    data=aoi_geojson,
    name=aoi_key,
    style={
        'opacity': 0.60, 'dashArray': '9', 'fillOpacity': 0.00, 'weight': 2, 'color': 'black',
    },
)
checkbox_layers_map[aoi_key] = geojson_layer
m.add(geojson_layer)

In [9]:
default_layer_key = "Habitat suitability weighted mean"

# Default colorbar
colorbar_widget = widgets.Image(
    value=colorbar_dict[checkbox_legend_map[default_layer_key]], format='png', 
    layout=Layout(object_fit='contain', margin='0px 0px 0px 0px'))

# Legend widget container
legend_container = widgets.Accordion(
    children=[colorbar_widget], titles=(f"Legend: {default_layer_key}",), 
    layout=Layout(max_width='350px', padding='0px 0px 0px 0px',))

# Track which layers are selected in a stack-like data structure
layer_stack = [default_layer_key]
# out is useful for debugging. We can capture the event outputs
# with the decorator.
out = widgets.Output()

@out.capture()
def layer_visible_switch(event):
    widget = event['owner']
    desc_id = widget.description
    print(desc_id)

    # Treat AOI as a basemap and don't include it in layer_stack
    if desc_id == aoi_key:
        layer_tile_url = checkbox_layers_map[desc_id]
        layer_tile_url.visible = event['new']    
            
    else:
        if event['new']:
            layer_stack.append(desc_id)
        else:
            layer_stack.pop(layer_stack.index(desc_id))
    
        layer_tile_url = checkbox_layers_map[desc_id]
        layer_tile_url.visible = event['new']
        cur_layers = m.layers
        # Preserve basemap in the back
        layer_order = [cur_layers[0]]
        
        for layer in cur_layers[1:]:
            # Skip AOI layer, we always want that in the foreground
            if layer.name == aoi_key:
                continue
            # We want the currently toggled layer, whether visible or not, to be last
            if layer.url != layer_tile_url.url:
                layer_order.append(layer)
        layer_order.append(layer_tile_url)
        # Add AOI last, to be in foreground
        layer_order.append(checkbox_layers_map[aoi_key])
        m.layers = layer_order

        print(layer_stack)
        if layer_stack:
            legend_id = layer_stack[-1]
            colorbar_widget.value = colorbar_dict[checkbox_legend_map[legend_id]]
            legend_container.titles = (f"Legend: {legend_id}",)
        else:
            legend_id = "Select layer"
            colorbar_widget.value = colorbar_dict['empty']
            legend_container.titles = (f"Legend: {legend_id}",)
                
widget_list = []
# Add AOI to accordion widget list
aoi_widget = widgets.Checkbox(
                value=True,
                description=geojson_layer.name,
                disabled=False,
                indent=False,
                layout=Layout(padding='0px 0px 0px 10px', margin='0px 0px 0px 0px', width='auto'),
            )
aoi_widget.observe(layer_visible_switch, names='value')
widget_list.append(aoi_widget)

for group_title, group_list in group_to_layers.items():
    group_header_widget = widgets.HTML(description="", value=f'<b>{group_title}<b>')
    checkbox_widget_group = []
    # Only expand the accordion menu w/ the active layer
    accordion_active = False
    for layer_key in group_list:
        
        layer_active = False
        # Default a layer to be active and visible
        if layer_key == default_layer_key:
            layer_active = True
            accordion_active = True
            
        tmp_widget = widgets.Checkbox(
                value=layer_active,
                description=layer_key,
                disabled=False,
                indent=False,
                layout=Layout(padding='0px 0px 0px 10px', margin='0px 0px 0px 0px', width='auto'),
            )
        tmp_widget.observe(layer_visible_switch, names='value')
        checkbox_widget_group.append(tmp_widget)
        checkbox_layers_map[layer_key].visible=layer_active
        m.add(checkbox_layers_map[layer_key])
    
    # Only expand the accordion menu w/ the active layer
    acc_idx = None
    if accordion_active:
        acc_idx = 0
    widget_list.append(widgets.Accordion(
                        children=[widgets.VBox(checkbox_widget_group)],
                        titles=(group_title,), selected_index=acc_idx))
    
vbox_widget = widgets.VBox(widget_list, layout=Layout(height='auto', width='auto', overflow_y='auto', padding='0px 0px 0px 0px'))
# Since we're defaulting a layer on, expand the accordion w/ select_index=0
layers_control_acc = widgets.Accordion(
    children=[vbox_widget], titles=("Layers",), 
    layout=Layout(max_height='350px', padding='0px 0px 0px 0px'),
    selected_index=0)

widget_layer_control = WidgetControl(widget=layers_control_acc, position="bottomleft")
widget_legend_control = WidgetControl(widget=legend_container, position="bottomright")

#map_hbox = widgets.HBox([m]) #, layout=Layout(height="70%"))
widgets.VBox([m, plot_grid])

In [10]:
%%capture
m.add(widget_layer_control)
# Add an inital legend control
m.add(widget_legend_control)

In [ ]:
#out